# 08 — China North Plain, SPOT 6/7 with Delineate-Anything

Delineates agricultural fields in the North China Plain using **SPOT 6/7** imagery at 6 m resolution and the **Delineate-Anything** engine.

> **Note:** SPOT 6/7 GEE access is restricted to select users (internal DRI use only). External users can contact the agribound author (sayantan.majumdar@dri.edu) to request field boundary processing.

**Estimated runtime:** ~15–30 minutes (1 year, 6 m resolution, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/china_north_plain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "spot"
ENGINE = "delineate-anything"
YEAR = 2022

## Create Study Area

AOI near Shijiazhuang, Hebei Province.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [114.40, 37.95],
                        [114.55, 37.95],
                        [114.55, 38.05],
                        [114.40, 38.05],
                        [114.40, 37.95],
                    ]
                ],
            },
            "properties": {"name": "North China Plain AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "north_china_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Delineation

In [ ]:
output_path = OUTPUT_DIR / f"fields_spot_{YEAR}.gpkg"

try:
    gdf = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=YEAR,
        engine=ENGINE,
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        cloud_cover_max=15,
        min_area=3000,
        simplify=2.0,
    )

    print(f"Delineated {len(gdf)} fields")
    if "metrics:area" in gdf.columns:
        area_ha = gdf["metrics:area"].sum() / 10000
        print(f"Total agricultural area: {area_ha:,.1f} ha")

except Exception as exc:
    print(f"SPOT access error: {exc}")
    print(
        "SPOT 6/7 is restricted to select GEE users. "
        "Contact the agribound author for processing assistance."
    )

## Visualization

In [ ]:
try:
    m = agribound.show_boundaries(
        gdf,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_north_china.html"),
    )
    m
except NameError:
    print("No results to visualize (SPOT access may be restricted).")